In [1]:
import torch
from model.model import load_model
from dataset.dataset import load_data, create_dataloaders
from functions.optimizer import load_optimizer
from functions.loss import load_loss_fun
from functions.chc_eval import celeba_train, celeba_eval, celeba_log_plot
from functions.xai import explain_dataset, evaluate_explainations, visualize_k_expl
from utils.utils import enable_reproducibility
from functions.loss import load_loss_fun
import matplotlib.pyplot as plt

In [ ]:
def wb_test(seed, loss_name, lr, epoch, reg_rate):
  use_cuda = torch.cuda.is_available()
  device = 'cuda' if use_cuda else 'cpu'
  enable_reproducibility(seed)

  model = load_model("ResNet", model_name="resnet50", n_classes=2, pretrained=True, device=device)
  optim = load_optimizer("SGD", model.parameters(), lr=lr)
  if loss_name == "CrossEntropy":
    loss = load_loss_fun(loss_name)
  else:
    loss = load_loss_fun("RRR", reg_rate = reg_rate, normalize=True)
  
  train_set, val_set, test_set = load_data(
    "Waterbirds", reload=False, balance=True, seed=seed)
  data = [train_set, val_set, test_set]
  params = {"batch_size":64}
  m_params = [params]*3
  train_loader, val_loader, test_loader = create_dataloaders(data, m_params)

  log, dyn = wb_train(
    model=model, 
    train_loader=train_loader, 
    optimizer=optim, 
    loss_fun=loss, 
    n_epochs=epoch, 
    eval_loader=val_loader, 
    #patience=3,
    device=device
  )
  print(log)


  all_attr, all_imgs = explain_dataset(train_loader, model, device)
  return all_attr, all_imgs, test_set


In [2]:
def chc_test(seed, loss_name, lr, epoch, reg_rate):
  use_cuda = torch.cuda.is_available()
  device = 'cuda' if use_cuda else 'cpu'
  enable_reproducibility(seed)

  model = load_model("ResNet", model_name="resnet50", n_classes=2, pretrained=True, device=device)
  optim = load_optimizer("SGD", model.parameters(), lr=lr)
  if loss_name == "CrossEntropy":
    loss = load_loss_fun(loss_name)
  else:
    loss = load_loss_fun("RRR", reg_rate = reg_rate, normalize=True)
  
  train_set, val_set, test_set = load_data("CelebAHC", reload=False)
  data = [train_set, val_set, test_set]
  params = {"batch_size":32}
  m_params = [params]*3
  train_loader, val_loader, test_loader = create_dataloaders(data, m_params)

  log, dyn = celeba_train(
    model=model, 
    train_loader=train_loader, 
    optimizer=optim, 
    loss_fun=loss, 
    n_epochs=epoch, 
    eval_loader=val_loader, 
    device=device
  )

  all_attr, all_imgs = explain_dataset(train_loader, model, device)
  return all_attr, all_imgs, train_set


In [3]:
all_attr, all_imgs, train_set = chc_test(123, "CrossEntropy", 1e-3, 10, 1)



Epoch 10/10: 100%|██████████| 10/10 [25:02<00:00, 150.28s/it, ce_loss=0.2767, acc=0.9266, val_loss=0.6057, val_acc=0.6667, worst_grp=0.1905]


In [ ]:
visualize_k_expl(all_attr, all_imgs, train_set, 0, k=3)